Corpus geral

In [ ]:
# Imports
import pandas as pd

# Carregando e visualizando CSV
df = pd.read_csv('database.csv')

df[["source", "target"]]

In [ ]:
# Criando corpus paralelo em formato txt
with open("zoe-por.txt", "w", encoding="utf-8") as corpus:
    for row in df.itertuples():
        source = row.source
        target = row.target

        corpus.write(f"{source}\t{target}\n")

Corpus separado

In [ ]:
# Imports
import pandas as pd

# Carregando e visualizando CSV
df = pd.read_csv('database.csv')
df.head()

In [ ]:
# Dados iniciais
print(f"Total de pares: {len(df)}")

In [ ]:
# Removendo espaços extras
df["source"] = df["source"].astype(str).str.strip()
df["target"] = df["target"].astype(str).str.strip()

In [ ]:
# Normalização
import re
import unicodedata

def normalizeString(s):
    s = unicodedata.normalize('NFC', s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s) # Identifica .!? e coloca um espaço ' ' antes
         # Remove qualquer coisa que não seja letra ou pontuação básica

    return s.strip()

In [ ]:
# Normalizando o corpus antes do filtro
df["source"] = df["source"].apply(normalizeString)
df["target"] = df["target"].apply(normalizeString)

In [ ]:
# Removendo duplicatas
antes = len(df)

df_limpo = df.drop_duplicates(subset=["source", "target"])

depois = len(df_limpo)

print(f"Duplicatas removidas: {antes - depois}")
print(f"Pares restantes: {depois}")

df = df_limpo

In [ ]:
# Aplicando filtro de comprimento máximo
MAX = 10

linhas_filtradas = []

antes_filtro = len(df)
qtd_removidos = 0

for linha in df.itertuples():
    source = linha.source
    target = linha.target 

    if len(str(source).split()) <= MAX and len(str(target).split()) <= MAX:
        linhas_filtradas.append({"source": source, "target": target})
    else: 
        qtd_removidos += 1

df_filtro = pd.DataFrame(linhas_filtradas)

print("Filtro comprimento max <= 10\n")
print(f"Pares antes do filtro: {antes_filtro}")
print(f"Pares após o filtro: {len(df_filtro)}")
print(f"Pares removidos: {qtd_removidos}")

df = df_filtro

In [ ]:
# Validando se filtro de comprimento máximo foi aplicado corretamente
maior_source = 0
maior_target = 0

for linha in df.itertuples():
    source = len(str(linha.source).split())
    target = len(str(linha.target).split())

    if source > maior_source:
        maior_source = source

    if target > maior_target:
        maior_target = target

print("Maior source:", maior_source)
print("Maior target:", maior_target)

In [ ]:
# Dividindo dataframe => treinamento, avaliacao e teste
# Lib responsável por fazer a divisão do dataframe
from sklearn.model_selection import train_test_split

# 80% para treinamento e 20% para uma base temporária
treinamento, tmp = train_test_split(
    df,
    train_size=0.8,
    random_state=42,
    shuffle=True # dados embaralhados antes da divisão
)

# dos 20% armazenados temporariamente, 50% para validação e 50% para teste 
validacao, teste = train_test_split(
    tmp,
    train_size=0.5,
    random_state=42,
    shuffle=True
)

In [ ]:
# Visualizando divisão do corpus
print("Divisão do corpus:\n")
print(f"Treinamento: {len(treinamento)} pares")
print(f"Validação: {len(validacao)} pares")
print(f"Teste: {len(teste)} pares")

print(f"Total: {len(treinamento) + len(validacao) + len(teste)} pares")

In [ ]:
# Salvando as 3 divisões do corpus
def salvar_corpus(df, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as corpus:
        for linha in df.itertuples():
            corpus.write(f"{linha.source}\t{linha.target}\n")

In [ ]:
# Salvando base de treinamento, validação e teste em txt
salvar_corpus(treinamento, "treinamento.txt")
salvar_corpus(validacao, "validacao.txt")
salvar_corpus(teste, "teste.txt")

In [ ]:
# Verificando os conjuntos
conjunto_treinamento = set(zip(treinamento["source"], treinamento["target"])) # => set([(source, target)]) => {(source, target)}
conjunto_validacao = set(zip(validacao["source"], validacao["target"]))
conjunto_teste = set(zip(teste["source"], teste["target"]))

print("Sobreposição de conjuntos:\n")
print(f"Treinamento x Validação: {len(conjunto_treinamento & conjunto_validacao)}")
print(f"Treinamento x Teste: {len(conjunto_treinamento & conjunto_teste)}")
print(f"Validação x Teste: {len(conjunto_validacao & conjunto_teste)}")

In [ ]:
# Verificando os conjuntos
source_treinamento = set(treinamento["source"])
source_validacao = set(validacao["source"])
source_teste = set(teste["source"])

print("Sobreposição de sentenças Zo'é:\n")
print(f"Treinamento x Validação: {len(source_treinamento & source_validacao)}")
print(f"Treinamento x Teste: {len(source_treinamento & source_teste)}")
print(f"Validação x Teste: {len(source_validacao & source_teste)}")

In [ ]:
# Verificando os conjuntos
target_treinamento = set(treinamento["target"])
target_validacao = set(validacao["target"])
target_teste = set(teste["target"])

print("Sobreposição de sentenças em Português:\n")
print(f"Treinamento x Validação: {len(target_treinamento & target_validacao)}")
print(f"Treinamento x Teste: {len(target_treinamento & target_teste)}")
print(f"Validação x Teste: {len(target_validacao & target_teste)}")

In [ ]:
# Visão geral dos dados obtidos
print("Visão Geral:\n")
print(f"Total de pares após limpeza: {len(df)}")
print(f"Treinamento: {len(treinamento)}")
print(f"Validação: {len(validacao)}")
print(f"Teste: {len(teste)}")

print()

print("Sobreposição de conjuntos:")
print(f"Treinamento x Validação: {len(conjunto_treinamento & conjunto_validacao)}")
print(f"Treinamento x Teste: {len(conjunto_treinamento & conjunto_teste)}")
print(f"Validação x Teste: {len(conjunto_validacao & conjunto_teste)}")

print()

print("Sobreposição de sentenças Zo'é:")
print(f"Treinamento x Validação: {len(source_treinamento & source_validacao)}")
print(f"Treinamento x Teste: {len(source_treinamento & source_teste)}")
print(f"Validação x Teste: {len(source_validacao & source_teste)}")
print()

print("Sobreposição de sentenças em Português:")
print(f"Treinamento x Validação: {len(target_treinamento & target_validacao)}")
print(f"Treinamento x Teste: {len(target_treinamento & target_teste)}")
print(f"Validação x Teste: {len(target_validacao & target_teste)}")